# 02c. 시너지 알파 선택 (AlphaGen 아이디어 적용)

**논문**: Yu et al., *Generating Synergistic Formulaic Alpha Collections via Reinforcement Learning* (KDD'23)

**차용한 핵심 아이디어** (전체 RL 파이프라인이 아니라 핵심 원리만 룰 기반으로 이식)
1. 알파를 **개별 IC**가 아니라 **기존 풀에 추가했을 때의 한계 기여도**로 채택 여부를 결정한다 (논문의 reward 설계).
2. 전통적인 mutual-IC 필터링(상관 높은 알파를 무조건 배제)은 실제 결합 성능과 어긋날 수 있다는 논문의 관찰(Q3)을 GAPS 데이터로 직접 검증한다.
3. 알파 결합 가중치는 매번 시계열을 재구성하지 않고, alpha 개별 IC + alpha 쌍 상호 IC만으로 MSE 최적 가중치를 구하는 논문 Theorem 3.1의 닫힌 해로 빠르게 계산한다.

**단순화한 부분**: 논문은 PPO로 무한한 수식 공간을 탐색하지만, 여기서는 이미 계산되어 있는 20개 기술지표로 만든 11개 후보 알파에 대해 탐욕적 전진 선택(greedy forward selection)으로 대체한다. 후보 풀이 작아 RL 없이도 전수 평가가 가능하기 때문.

**검증 방법**: 188개 ETF 단면에서 20거래일 선행 수익률과의 Pearson IC를 계산해 (1) 단일 최고 알파, (2) top-k 선택, (3) mutual-IC 필터 선택, (4) 시너지 탐욕 선택 네 가지를 train/test로 비교한다. 마지막으로 시너지 알파 조합 점수를 월간 집계해, `02_backtest_momentum.ipynb`의 "카테고리 내 3개월 모멘텀 ETF 선택"을 대체하는 백테스트까지 수행한다.

**주의**: 카테고리(10개) 단위가 아니라 **ETF 단위(188개)**에서 단면 IC를 계산한다 — 원 논문처럼 단면 크기가 충분해야 상관계수가 통계적으로 의미가 있기 때문 (카테고리 10개로는 n이 너무 작아 사용 불가).


In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import sys, os
import warnings
warnings.filterwarnings('ignore')

# ── 환경 감지 (Colab vs 로컬) ──────────────────────────────────────────────────
IN_COLAB = 'google.colab' in sys.modules

if IN_COLAB:
    from google.colab import drive
    drive.mount('/content/drive')
    DRIVE_BASE = '/content/drive/MyDrive/졸업프로젝트/GAPS_대회'
    DATA_DIR   = '/content/drive/MyDrive/졸업프로젝트/GAPS_대회/data'
    OUT_DIR    = f'{DRIVE_BASE}/data'
    SRC_DIR    = f'{DRIVE_BASE}/src'
    os.system('apt-get install -y fonts-nanum > /dev/null 2>&1')
    import matplotlib.font_manager as fm
    fm._load_fontmanager(try_read_cache=False)
    plt.rcParams['font.family'] = 'NanumGothic'
else:
    DATA_DIR = '/Users/seungbin/school/4-1/졸업프로젝트/data'
    OUT_DIR  = '/Users/seungbin/school/4-1/졸업프로젝트/GAPS_대회/data'
    SRC_DIR  = '/Users/seungbin/school/4-1/졸업프로젝트/GAPS_대회/src'
    plt.rcParams['font.family'] = 'AppleGothic'

plt.rcParams['axes.unicode_minus'] = False
plt.rcParams['figure.dpi'] = 120
os.makedirs(OUT_DIR, exist_ok=True)

print(f'환경: {"Colab" if IN_COLAB else "로컬"}')
print(f'DATA_DIR : {DATA_DIR}')
print(f'OUT_DIR  : {OUT_DIR}')
print('※ 01_collect_etf_data_colab.ipynb 가 저장한 etf_ohlcv_indicators.parquet(long format)가 DATA_DIR에 있어야 합니다.')


In [ ]:
sys.path.append(SRC_DIR)
from etf_universe import ETF_LIST, RISK_CATEGORIES, SAFE_CATEGORIES

ticker2cat  = {e['ticker']: e['category']  for e in ETF_LIST}
ticker2risk = {e['ticker']: e['is_risk']   for e in ETF_LIST}
ticker2name = {e['ticker']: e['name']      for e in ETF_LIST}
ALL_CATEGORIES = RISK_CATEGORIES + SAFE_CATEGORIES

print('위험 카테고리:', RISK_CATEGORIES)
print('안전 카테고리:', SAFE_CATEGORIES)


In [ ]:
# 01번 노트북이 저장한 long format (OHLCV + 20개 기술지표)
df_long = pd.read_parquet(f'{DATA_DIR}/etf_ohlcv_indicators.parquet')
df_long['Date'] = pd.to_datetime(df_long['Date'])
df_long = df_long.sort_values(['ticker', 'Date'])

print(f'Long format: {df_long.shape}, 종목수={df_long["ticker"].nunique()}, '
      f'기간={df_long["Date"].min().date()} ~ {df_long["Date"].max().date()}')

def pivot_col(col: str) -> pd.DataFrame:
    """long format → Date x ticker wide format"""
    return df_long.pivot(index='Date', columns='ticker', values=col).sort_index()

RAW = {
    col: pivot_col(col)
    for col in ['Close', 'RSI_14', 'MACD_hist', 'BB_upper', 'BB_lower',
                'SMA_20', 'SMA_60', 'EMA_12', 'EMA_26',
                'Volume_ratio', 'Volatility_20d', 'ATR_14']
}
print('피벗 완료:', {k: v.shape for k, v in RAW.items()})


## 1. 후보 formulaic alpha 정의

AlphaGen 논문(Table 1, 4)의 연산자(CS-*, TS-*)를 본따, 기존 20개 기술지표 위에
간단한 수식으로 11개 후보 알파를 정의한다. RL 탐색 대신 사람이 직접 후보 풀을
구성한 '간이 버전'이며, 이후 단계에서 논문의 핵심 아이디어(풀에 대한 한계 기여도로
채택)를 그대로 적용한다.

In [ ]:
def build_alpha_candidates(raw: dict) -> dict:
    close = raw['Close']
    alphas = {}
    alphas['mom_3m']          = close / close.shift(60) - 1
    alphas['mom_12m']         = close / close.shift(250) - 1
    alphas['reversal_5d']     = -(close / close.shift(5) - 1)
    alphas['rsi_meanrev']     = 50 - raw['RSI_14']
    alphas['macd_hist_norm']  = raw['MACD_hist'] / close
    alphas['bb_position']     = (close - raw['BB_lower']) / (raw['BB_upper'] - raw['BB_lower']) - 0.5
    alphas['sma_cross']       = raw['SMA_20'] / raw['SMA_60'] - 1
    alphas['ema_cross']       = raw['EMA_12'] / raw['EMA_26'] - 1
    alphas['vol_ratio_signal'] = raw['Volume_ratio'] - 1
    alphas['low_vol_carry']   = -raw['Volatility_20d']
    alphas['atr_norm']        = -(raw['ATR_14'] / close)
    return alphas

ALPHA_RAW = build_alpha_candidates(RAW)
CANDIDATES = list(ALPHA_RAW.keys())
print(f'{len(CANDIDATES)}개 후보 알파:', CANDIDATES)


## 2. 예측 대상 및 단면 IC 계산 함수

논문과 동일하게 20거래일(약 1개월) 선행 수익률을 타깃으로 하고, 하루 단면(=그 날
188개 ETF)에서 알파값과 타깃의 Pearson 상관계수를 구한 뒤 전체 기간 평균을 IC로
사용한다 (Eq. 1~2). 상관계수는 스케일에 불변이므로 개별/상호 IC 계산 시에는 별도
정규화가 필요 없다 (정규화는 이후 여러 알파를 가중합할 때만 필요).

In [ ]:
HORIZON = 20  # 선행 수익률 기간 (거래일)
target_raw = RAW['Close'].shift(-HORIZON) / RAW['Close'] - 1

MIN_OBS = 30  # 하루 단면에서 최소 유효 ETF 수

def daily_ic_series(a: pd.DataFrame, b: pd.DataFrame, min_obs: int = MIN_OBS) -> pd.Series:
    """일자별 단면 Pearson IC (두 wide DataFrame 사이, 컬럼=티커가 단면)"""
    idx = a.index.intersection(b.index)
    cols = a.columns.intersection(b.columns)
    a2, b2 = a.loc[idx, cols], b.loc[idx, cols]
    ic = a2.corrwith(b2, axis=1)
    valid_counts = (a2.notna() & b2.notna()).sum(axis=1)
    ic = ic.where(valid_counts >= min_obs)
    return ic.dropna()

def cross_sectional_normalize(df: pd.DataFrame, min_obs: int = MIN_OBS) -> pd.DataFrame:
    """논문의 N 연산자: 일자별 단면 평균 0, 길이 1로 정규화 (알파 결합 시에만 사용)"""
    valid_counts = df.notna().sum(axis=1)
    mean = df.mean(axis=1)
    centered = df.sub(mean, axis=0)
    norm = np.sqrt((centered ** 2).sum(axis=1))
    z = centered.div(norm, axis=0)
    z.loc[valid_counts < min_obs, :] = np.nan
    return z

def slice_period(df: pd.DataFrame, start: str, end: str) -> pd.DataFrame:
    return df.loc[start:end]


In [ ]:
# ── Train / Test 분할 (GAPS 백테스트와 동일하게 워밍업 포함 학습 → 2024~ 테스트) ──
TRAIN_START, TRAIN_END = '2020-01-01', '2023-12-31'
TEST_START,  TEST_END  = '2024-01-01', '2026-05-31'

target_train = slice_period(target_raw, TRAIN_START, TRAIN_END)
target_test  = slice_period(target_raw, TEST_START,  TEST_END)

ic_rows = []
for name in CANDIDATES:
    a_train = slice_period(ALPHA_RAW[name], TRAIN_START, TRAIN_END)
    a_test  = slice_period(ALPHA_RAW[name], TEST_START,  TEST_END)
    ic_rows.append({
        'alpha': name,
        'IC_train': daily_ic_series(a_train, target_train).mean(),
        'IC_test':  daily_ic_series(a_test,  target_test).mean(),
    })

ic_table = pd.DataFrame(ic_rows).set_index('alpha').sort_values('IC_train', ascending=False)
print('=== 개별 알파 IC (20거래일 선행 수익률, 188개 ETF 단면) ===')
print(ic_table.round(4).to_string())


In [ ]:
# ── 알파 간 평균 상호 IC 행렬 (train) ──────────────────────────────────────────
mutual_ic = pd.DataFrame(index=CANDIDATES, columns=CANDIDATES, dtype=float)
for i, a in enumerate(CANDIDATES):
    for j, b in enumerate(CANDIDATES):
        if j < i:
            mutual_ic.loc[a, b] = mutual_ic.loc[b, a]
        elif j == i:
            mutual_ic.loc[a, b] = 1.0
        else:
            fa = slice_period(ALPHA_RAW[a], TRAIN_START, TRAIN_END)
            fb = slice_period(ALPHA_RAW[b], TRAIN_START, TRAIN_END)
            mutual_ic.loc[a, b] = daily_ic_series(fa, fb).mean()

print('=== 알파 간 평균 상호 IC (train) ===')
print(mutual_ic.round(2).to_string())


## 3. 결합 모델 (Theorem 3.1) + 시너지 기반 탐욕 선택

논문 Theorem 3.1: 정규화된 알파들의 선형결합에 대한 MSE 손실은 개별 IC(`σ̄_y(f_i)`)와
알파 쌍 상호 IC(`σ̄(f_i,f_j)`)만으로 표현되므로, 매번 시계열을 재구성하지 않고도
최적 가중치를 `w* = Σ_mut⁻¹ · σ_y` 형태의 닫힌 해로 구할 수 있다. (수치 안정성을 위해
소량의 ridge 항을 더한다.)

탐욕 선택은 논문 Algorithm 2의 reward 정의(= 새 알파를 풀에 추가했을 때 재계산한
실제 결합 IC, `IC_new = σ̄_y(Σ w_i f_i)`)를 그대로 사용하되, RL 대신 매 스텝마다
남은 후보 전체를 brute-force로 평가해 최댓값을 선택한다.

In [ ]:
sigma_y = ic_table['IC_train']  # Theorem 3.1의 sigma_y

def solve_combination_weights(alpha_names: list, ridge: float = 1e-3) -> pd.Series:
    """Theorem 3.1의 닫힌 해: MSE 최소화 가중치 = Sigma_mut^{-1} sigma_y (ridge로 안정화)"""
    sy = sigma_y.loc[alpha_names].values
    Sigma = mutual_ic.loc[alpha_names, alpha_names].values.astype(float)
    Sigma_reg = Sigma + ridge * np.eye(len(alpha_names))
    w = np.linalg.solve(Sigma_reg, sy)
    return pd.Series(w, index=alpha_names)

def build_composite(alpha_names: list, weights: pd.Series, start: str, end: str) -> pd.DataFrame:
    """선택된 알파들을 일자별 단면 정규화 후 가중합 → mega-alpha"""
    composite = None
    for name in alpha_names:
        z = cross_sectional_normalize(slice_period(ALPHA_RAW[name], start, end))
        term = z * weights[name]
        composite = term if composite is None else composite.add(term, fill_value=0)
    return composite

def evaluate_pool_ic(alpha_names: list, weights: pd.Series, start: str, end: str) -> float:
    """Algorithm 2의 IC_new를 실제 데이터로 재계산 (근사식이 아닌 실측값)"""
    if not alpha_names:
        return 0.0
    composite = build_composite(alpha_names, weights, start, end)
    tgt = slice_period(target_raw, start, end)
    return daily_ic_series(composite, tgt).mean()


In [ ]:
def greedy_synergistic_selection(candidates: list, max_size: int = 8, min_gain: float = 0.0005):
    """AlphaGen 핵심 아이디어의 룰 기반 버전:
    RL 리워드(= 새 알파를 풀에 추가했을 때의 실제 결합 IC 개선분) 대신,
    작은 후보 풀 전체를 매 스텝 brute-force로 평가해 최댓값을 선택한다."""
    selected, history = [], []
    current_ic = 0.0
    remaining = list(candidates)

    while remaining and len(selected) < max_size:
        best_name, best_ic, best_w = None, current_ic, None
        for cand in remaining:
            trial = selected + [cand]
            w = solve_combination_weights(trial)
            ic = evaluate_pool_ic(trial, w, TRAIN_START, TRAIN_END)
            if ic > best_ic:
                best_name, best_ic, best_w = cand, ic, w
        if best_name is None or (best_ic - current_ic) < min_gain:
            break
        history.append({'step': len(selected) + 1, 'added': best_name,
                         'pool_ic': best_ic, 'gain': best_ic - current_ic})
        selected.append(best_name)
        remaining.remove(best_name)
        current_ic = best_ic

    final_w = solve_combination_weights(selected) if selected else pd.Series(dtype=float)
    return selected, final_w, pd.DataFrame(history)

synergy_selected, synergy_weights, synergy_history = greedy_synergistic_selection(CANDIDATES)
print('=== 시너지 기반 탐욕 선택 이력 ===')
print(synergy_history.round(4).to_string(index=False))
print('\n최종 선택 알파:', synergy_selected)
print('최종 가중치:')
print(synergy_weights.round(3).to_string())


## 4. 베이스라인과 비교 (논문 Q1/Q3 검증)

논문이 기존 GP 계열에서 흔히 쓰던 두 방식(top-k, mutual-IC 필터)이 실제 결합
성능과 어긋날 수 있음을 보였던 것과 동일한 비교를 GAPS 데이터로 재현한다.

In [ ]:
K = max(len(synergy_selected), 1)  # 공정 비교를 위해 동일한 풀 크기 사용

# baseline 1: top-k (개별 IC 상위 k개)
topk_selected = ic_table.sort_values('IC_train', ascending=False).head(K).index.tolist()
topk_weights  = solve_combination_weights(topk_selected)

# baseline 2: mutual-IC filter (개별 IC 내림차순으로 보되, 이미 뽑힌 알파와
# |mutual IC| > 0.7이면 skip — 전통적인 '다양성' 필터링 방식)
FILTER_THRESHOLD = 0.7
filter_selected = []
for name in ic_table.sort_values('IC_train', ascending=False).index:
    if len(filter_selected) >= K:
        break
    if all(abs(mutual_ic.loc[name, s]) <= FILTER_THRESHOLD for s in filter_selected):
        filter_selected.append(name)
filter_weights = solve_combination_weights(filter_selected)

print('top-k 선택         :', topk_selected)
print('mutual-IC filter 선택:', filter_selected)
print('시너지 탐욕 선택     :', synergy_selected)


In [ ]:
def summarize(name, sel, w):
    return {
        '방법': name,
        '알파 수': len(sel),
        'Train IC': evaluate_pool_ic(sel, w, TRAIN_START, TRAIN_END),
        'Test IC (OOS)': evaluate_pool_ic(sel, w, TEST_START, TEST_END),
    }

best_single = ic_table['IC_train'].idxmax()
best_single_w = pd.Series({best_single: 1.0})

compare_table = pd.DataFrame([
    summarize('단일 최고 알파', [best_single], best_single_w),
    summarize(f'top-{K} (개별 IC 순)', topk_selected, topk_weights),
    summarize(f'mutual-IC filter (k={K}, thr={FILTER_THRESHOLD})', filter_selected, filter_weights),
    summarize(f'시너지 탐욕 선택 (k={K}, 논문 방식)', synergy_selected, synergy_weights),
]).set_index('방법')

print('=== 알파 결합 방식 비교 (AlphaGen Table 2 스타일) ===')
print(compare_table.round(4).to_string())

compare_table.to_csv(f'{OUT_DIR}/synergistic_alpha_comparison.csv', encoding='utf-8-sig')
print(f'\n저장: {OUT_DIR}/synergistic_alpha_comparison.csv')


## 5. 월간 합성 알파(mega-alpha) 저장

시너지 탐욕 선택으로 얻은 알파 조합을 전체 기간에 적용해 월말 기준으로 집계한다.
이 점수는 `02_backtest_momentum.ipynb`의 "카테고리 내 3개월 모멘텀 ETF 선택"을
대체하는 신호로 아래 6장에서 사용한다.

In [ ]:
composite_full = build_composite(
    synergy_selected, synergy_weights,
    RAW['Close'].index.min().strftime('%Y-%m-%d'),
    RAW['Close'].index.max().strftime('%Y-%m-%d'),
)
composite_monthly = composite_full.resample('ME').last()

composite_path = f'{OUT_DIR}/alpha_composite_monthly.parquet'
composite_monthly.to_parquet(composite_path)
print(f'저장: {composite_path}  shape={composite_monthly.shape}')

report = synergy_weights.rename('weight').to_frame()
report['IC_train_individual'] = ic_table.loc[synergy_selected, 'IC_train']
report['IC_test_individual']  = ic_table.loc[synergy_selected, 'IC_test']
report_path = f'{OUT_DIR}/synergistic_alpha_report.csv'
report.to_csv(report_path, encoding='utf-8-sig')
print(f'저장: {report_path}')
print(report.round(4).to_string())


## 6. 백테스트: 시너지 알파로 ETF 선택 (기존 3개월 모멘텀 대체)

카테고리 선택(12개월 모멘텀, 위험 65%/안전 35%, 상위 3/2개 카테고리)은
`02_backtest_momentum.ipynb`와 완전히 동일한 로직을 재사용하고,
**카테고리 내 ETF 선택만** 3개월 모멘텀 랭킹 대신 시너지 알파 합성 점수 랭킹으로
교체한다. 나머지 파라미터(TOP_ETF_FRAC=33%, 거래비용 0.1%)도 동일하게 유지해
02번 노트북의 결과와 직접 비교 가능하게 한다.

In [ ]:
close = RAW['Close'].ffill(limit=5)
monthly = close.resample('ME').last()
monthly = monthly.loc['2020-01-31':]
monthly_ret = monthly.pct_change().iloc[1:]

def get_category_returns(ret_df: pd.DataFrame) -> pd.DataFrame:
    cat_ret = {}
    for cat in ALL_CATEGORIES:
        tickers = [t for t in ret_df.columns if ticker2cat.get(t) == cat]
        if tickers:
            cat_ret[cat] = ret_df[tickers].mean(axis=1)
    return pd.DataFrame(cat_ret)

cat_monthly_ret   = get_category_returns(monthly_ret)
cat_monthly_price = (1 + cat_monthly_ret).cumprod()

LOOKBACK_L   = 12    # 카테고리 선택: 12개월 모멘텀 (02번과 동일)
RISK_BUDGET  = 0.65
SAFE_BUDGET  = 0.35
TOP_RISK     = 3
TOP_SAFE     = 2
TOP_ETF_FRAC = 0.33  # 카테고리 내 상위 ETF 비율 (02번과 동일)
COST         = 0.001

def compute_momentum_signal(cat_price: pd.DataFrame, date_idx: int, lb: int) -> pd.Series:
    if date_idx < lb:
        return pd.Series(dtype=float)
    return (cat_price.iloc[date_idx] / cat_price.iloc[date_idx - lb] - 1).dropna()

def compute_cat_weights(momentum: pd.Series) -> dict:
    risk_mom = momentum[momentum.index.isin(RISK_CATEGORIES)].sort_values(ascending=False)
    safe_mom = momentum[momentum.index.isin(SAFE_CATEGORIES)].sort_values(ascending=False)
    weights = {}
    for cat in risk_mom.head(TOP_RISK).index:
        weights[cat] = RISK_BUDGET / TOP_RISK
    for cat in safe_mom.head(TOP_SAFE).index:
        weights[cat] = SAFE_BUDGET / TOP_SAFE
    return weights

def expand_to_etf_weights_alpha(cat_weights: dict, available_tickers: list,
                                 alpha_monthly: pd.DataFrame, date) -> pd.Series:
    """02번 노트북의 '카테고리 내 3개월 모멘텀 랭킹'을 시너지 알파 점수 랭킹으로 교체"""
    if date not in alpha_monthly.index:
        return pd.Series(dtype=float)
    scores_all = alpha_monthly.loc[date]

    etf_w = {}
    for cat, w in cat_weights.items():
        members = [t for t in available_tickers if ticker2cat.get(t) == cat
                   and t in alpha_monthly.columns]
        if not members:
            continue
        scores = scores_all[members].dropna()
        n_sel = max(1, int(len(members) * TOP_ETF_FRAC))
        selected = scores.sort_values(ascending=False).head(n_sel).index.tolist() if len(scores) else members
        for t in selected:
            etf_w[t] = w / len(selected)

    total = sum(etf_w.values())
    return pd.Series({k: v / total for k, v in etf_w.items()}) if total > 0 else pd.Series(dtype=float)


# ── 백테스트 루프 ──────────────────────────────────────────────────────────────
backtest_start = '2021-01-31'
bt_dates = cat_monthly_price.loc[backtest_start:].index

portfolio_values, monthly_returns_bt, turnover_log = [1.0], [], []
prev_weights = pd.Series(dtype=float)

for i, date in enumerate(bt_dates[:-1]):
    idx = cat_monthly_price.index.get_loc(date)
    momentum = compute_momentum_signal(cat_monthly_price, idx, LOOKBACK_L)
    if momentum.empty:
        portfolio_values.append(portfolio_values[-1])
        monthly_returns_bt.append(0.0)
        continue

    cat_w = compute_cat_weights(momentum)
    avail = monthly_ret.columns[monthly_ret.loc[bt_dates[i + 1]].notna()].tolist()
    etf_w = expand_to_etf_weights_alpha(cat_w, avail, composite_monthly, date)
    if etf_w.empty:
        portfolio_values.append(portfolio_values[-1])
        monthly_returns_bt.append(0.0)
        continue

    turnover = (etf_w.reindex(prev_weights.index, fill_value=0) - prev_weights).abs().sum() / 2
    cost = float(turnover) * COST
    turnover_log.append({'date': bt_dates[i + 1], 'turnover': float(turnover)})

    next_ret = monthly_ret.loc[bt_dates[i + 1]]
    port_ret = float((etf_w * next_ret.reindex(etf_w.index, fill_value=0)).sum()) - cost

    portfolio_values.append(portfolio_values[-1] * (1 + port_ret))
    monthly_returns_bt.append(port_ret)
    prev_weights = etf_w

alpha_port_series     = pd.Series(portfolio_values, index=bt_dates[:len(portfolio_values)])
alpha_port_ret_series = pd.Series(monthly_returns_bt, index=bt_dates[1:len(portfolio_values)])
alpha_turnover_df      = pd.DataFrame(turnover_log).set_index('date')

print(f'백테스트 완료: {bt_dates[0].date()} ~ {bt_dates[-1].date()}')
print(f'평균 월 회전율: {alpha_turnover_df["turnover"].mean()*100:.1f}%  (대회 기준 ≥10%)')


In [ ]:
def calc_metrics(ret_series: pd.Series, name: str) -> dict:
    n_months = len(ret_series)
    total_ret = (1 + ret_series).prod() - 1
    cagr = (1 + total_ret) ** (12 / n_months) - 1
    vol  = ret_series.std() * np.sqrt(12)
    sharpe = cagr / vol if vol > 0 else 0
    cum = (1 + ret_series).cumprod()
    dd = (cum / cum.cummax() - 1)
    mdd = dd.min()
    win_rate = (ret_series > 0).mean()
    return {
        '전략명':     name,
        '누적수익률':  f'{total_ret*100:.1f}%',
        'CAGR':      f'{cagr*100:.1f}%',
        '변동성(연)':  f'{vol*100:.1f}%',
        'Sharpe':    f'{sharpe:.2f}',
        'MDD':       f'{mdd*100:.1f}%',
        '월수익률>0':  f'{win_rate*100:.0f}%',
    }

alpha_metrics = calc_metrics(alpha_port_ret_series, '시너지 알파 ETF선택 (신규)')

# 02번 노트북이 저장해 둔 기존 결과와 비교
prev_metrics = pd.read_csv(f'{OUT_DIR}/backtest_metrics.csv', index_col=0)
compare_backtest = pd.concat([
    pd.DataFrame([alpha_metrics]).set_index('전략명'),
    prev_metrics,
])

print('=== 백테스트 성과 비교: 시너지 알파 ETF선택 vs 기존 전략들 ===')
print(compare_backtest.to_string())

compare_backtest.to_csv(f'{OUT_DIR}/backtest_metrics_with_alpha.csv', encoding='utf-8-sig')
alpha_port_ret_series.to_frame('alpha_synergy_ret').to_csv(
    f'{OUT_DIR}/backtest_returns_alpha_synergy.csv', encoding='utf-8-sig')
print('\n저장: backtest_metrics_with_alpha.csv, backtest_returns_alpha_synergy.csv')


In [ ]:
prev_returns = pd.read_csv(f'{OUT_DIR}/backtest_returns.csv', index_col=0, parse_dates=True)
momentum_series = (1 + prev_returns['momentum_ret']).cumprod()

fig, ax = plt.subplots(figsize=(12, 6))
ax.plot(alpha_port_series.index, alpha_port_series.values, lw=2.5, color='#E63946',
        label='시너지 알파 ETF선택 (신규)')
ax.plot(momentum_series.index, momentum_series.values, lw=1.8, color='#457B9D', ls='--',
        label='카테고리 모멘텀 (기존, 02번)')
ax.axhline(1, color='gray', lw=0.8, ls='--', alpha=0.5)
ax.set_ylabel('누적 수익률 (기준=1.0)')
ax.set_title('시너지 알파 ETF선택 vs 기존 카테고리 모멘텀 전략', fontsize=13, fontweight='bold')
ax.legend(loc='upper left', fontsize=9)
ax.grid(alpha=0.3)
plt.tight_layout()
plt.savefig(f'{OUT_DIR}/backtest_cumulative_alpha_vs_momentum.png', dpi=150, bbox_inches='tight')
plt.show()
print('저장: backtest_cumulative_alpha_vs_momentum.png')


## 7. 결론 및 다음 단계

- 4장의 비교표(`synergistic_alpha_comparison.csv`)가 시너지 탐욕 선택이 top-k /
  mutual-IC filter보다 결합 IC(train, test 모두)가 높게 나오는지 확인한다.
  만약 그렇지 않다면 후보 알파 풀이 너무 상관도가 높거나 단조로운 것이므로
  후보군을 늘리는 방향(변동성 국면별 alpha, 카테고리 간 상대모멘텀 등)을 검토한다.
- 6장의 백테스트 결과가 기존 `카테고리 모멘텀 (우리)` 대비 Sharpe/MDD를
  개선했다면, `02_backtest_momentum.ipynb`의 `expand_to_etf_weights` 함수를
  이 노트북의 `expand_to_etf_weights_alpha`로 교체하는 안을 정식 파이프라인에
  반영할 수 있다.
- 저장된 `alpha_composite_monthly.parquet`는 카테고리 레벨로 집계(`groupby(category).mean()`)해
  `02b_etf_hybrid_model_colab.ipynb`의 `ai_pred`와 앙상블하거나, RL 에이전트의
  state(150-dim)에 해석 가능한 추가 피처로 넣는 것도 가능하다.